<div dir="rtl" style="text-align: right;">

## ۹. سوال ۱-الف: مدل خام و همخطی کامل

مدل سوال را روی داده‌ی خام برازش می‌دهم، بدون هیچ حذف یا تغییری:

`monthly_spend ~ age + income + tenure_months + n_products + savings + investments + total_assets + region`

ماتریس طرح X شامل عرض از مبدأ، ۷ متغیر عددی و ۲ ستون dummy برای `region` است. پس ۱۰ ستون دارد. اگر همه‌ی ستون‌ها مستقل باشند، رتبه ۱۰ می‌شود.

</div>

In [2]:
import statsmodels.formula.api as smf

formula_raw = ('monthly_spend ~ age + income + tenure_months + n_products '
               '+ savings + investments + total_assets + C(region)')
m_raw = smf.ols(formula_raw, data=df).fit()

X = m_raw.model.exog
names = m_raw.model.exog_names
n, p = X.shape

print(f'n = {n}, p (columns of X) = {p}')
print(f'rank(X) = {np.linalg.matrix_rank(X)}')

NameError: name 'df' is not defined

<div dir="rtl" style="text-align: right;">

**توضیح:** ماتریس طرح ۱۰ ستون دارد ولی رتبه‌اش فقط ۹ است. یعنی یک ستون ترکیب خطی بقیه است و `X'X` معکوس‌پذیر نیست. با این حال نرم‌افزار خطا نمی‌دهد و عدد می‌دهد. در ادامه می‌بینیم این عددها قابل اعتماد نیستند.

</div>

In [2]:
eigvals, eigvecs = np.linalg.eigh(X.T @ X)

print('eigenvalues of X\'X (smallest 4):')
print(eigvals[:4])
print()
print('largest eigenvalue:', f'{eigvals[-1]:.3e}')
print('condition number of X:', f'{np.linalg.cond(X):.3e}')
print()
print('eigenvector of the smallest eigenvalue:')
print(pd.Series(eigvecs[:, 0], index=names).round(3))

eigenvalues of X'X (smallest 4):
[-1.81529680e-09  1.27895655e+01  5.19900241e+01  1.16607526e+02]

largest eigenvalue: 6.183e+07
condition number of X: 5.423e+15

eigenvector of the smallest eigenvalue:
Intercept            -0.000
C(region)[T.Metro]    0.000
C(region)[T.Other]    0.000
age                   0.000
income                0.000
tenure_months        -0.000
n_products            0.000
savings              -0.577
investments          -0.577
total_assets          0.577
dtype: float64


<div dir="rtl" style="text-align: right;">

**توضیح:** کوچک‌ترین مقدار ویژه‌ی `X'X` عملاً صفر است (حدود ۱e-9 و حتی منفی، که فقط خطای گرد کردن کامپیوتر است). عدد شرط ماتریس حدود ۵٫۴e15 است. بردار ویژه‌ی متناظر فقط روی سه ستون مقدار دارد: `savings` و `investments` با وزن ‎−۰٫۵۷۷ و `total_assets` با وزن ‎+۰٫۵۷۷، و بقیه صفرند. یعنی رابطه‌ی خطی دقیق این است:

`total_assets − savings − investments = 0`

این همان چیزی است که در بخش ۴ دیدیم: `total_assets` همیشه برابر جمع دو ستون دیگر است.

</div>

In [3]:
pd.DataFrame({'coef': m_raw.params, 'se': m_raw.bse}).round(4)

,coef,se
Intercept,4.6857,1.3661
C(region)[T.Metro],-5.5405,0.6027
C(region)[T.Other],-6.6684,0.5971
age,-0.0734,0.0300
income,0.1777,0.1283
tenure_months,0.0256,0.0127
n_products,0.5916,0.1591
savings,-0.0049,0.0144
investments,0.0050,0.0079
total_assets,0.0001,0.0074


In [4]:
v = eigvecs[:, 0] / eigvecs[:, 0][names.index('total_assets')]
beta_alt = m_raw.params.values + 5 * v
fitted_alt = X @ beta_alt

print('max |fitted_original - fitted_shifted|:', np.abs(m_raw.fittedvalues.values - fitted_alt).max())
print('coef of savings      -> original:', round(m_raw.params['savings'], 4),
      ' shifted:', round(beta_alt[names.index('savings')], 4))
print('coef of total_assets -> original:', round(m_raw.params['total_assets'], 4),
      ' shifted:', round(beta_alt[names.index('total_assets')], 4))

max |fitted_original - fitted_shifted|: 1.3629719575192212e-10
coef of savings      -> original: -0.0049  shifted: -5.0049
coef of total_assets -> original: 0.0001  shifted: 5.0001


<div dir="rtl" style="text-align: right;">

**توضیح:** وقتی یک جهت صفر در `X'X` داریم، می‌توان به ضرایب هر مقدار دلخواه از آن جهت را اضافه کرد و مقدارهای برازش‌شده کاملاً یکسان می‌مانند. اینجا ضرایب را جابه‌جا کردم و اختلاف مقدارهای برازش‌شده صفر (در حد خطای گرد کردن) شد، ولی ضریب `savings` و `total_assets` عوض شد. نتیجه: بی‌نهایت جواب برای ضرایب وجود دارد و فقط پیش‌بینی ثابت است.

عددهایی که نرم‌افزار چاپ می‌کند یکی از همین بی‌نهایت جواب است. `statsmodels` از شبه‌معکوس استفاده می‌کند و جواب با کوچک‌ترین نُرم را برمی‌گرداند. پس این ضرایب و خطاهای استاندارد را نمی‌شود تفسیر کرد.

</div>

<div dir="rtl" style="text-align: right;">

### اصلاح مدل

برای رفع همخطی کامل، `total_assets` را حذف می‌کنم. چون مقدار آن دقیقاً از `savings` و `investments` ساخته می‌شود، حذفش هیچ اطلاعاتی را از دست نمی‌دهد. هر کدام از سه ستون را می‌شد حذف کرد، ولی `total_assets` انتخاب طبیعی است چون ستون مشتق‌شده است.

در سوال ۱-ب می‌بینیم که `income` و `savings` هم تقریباً یکی هستند. برای همین مدل نهایی بعد از آن مرحله `savings` را هم از دست می‌دهد و همان مدل «بدون `total_assets` و `savings`» می‌شود که سوال ۲ از آن استفاده می‌کند.

</div>

In [5]:
formula_fix = ('monthly_spend ~ age + income + tenure_months + n_products '
               '+ savings + investments + C(region)')
m_fix = smf.ols(formula_fix, data=df).fit()
X_fix = m_fix.model.exog

def summarize(model, X_mat):
    rank = np.linalg.matrix_rank(X_mat)
    return {
        'columns (p)': X_mat.shape[1],
        'rank(X)': rank,
        'nominal df = n - p': n - X_mat.shape[1],
        'true df = n - rank': n - rank,
        'smallest eigenvalue': f'{np.linalg.eigvalsh(X_mat.T @ X_mat)[0]:.2e}',
        'condition number': f'{np.linalg.cond(X_mat):.2e}',
        'RSS': round(model.ssr, 2),
        'R2': round(model.rsquared, 4),
    }

pd.DataFrame({
    'raw model': summarize(m_raw, X),
    'without total_assets': summarize(m_fix, X_fix),
})

,raw model,without total_assets
columns (p),10,9
rank(X),9,9
nominal df = n - p,390,391
true df = n - rank,391,391
smallest eigenvalue,-1.82e-09,1.28e+01
condition number,5.42e+15,1.44e+03
RSS,9555.01,9555.01
R2,0.4408,0.4408


In [6]:
pd.DataFrame({'coef': m_fix.params, 'se': m_fix.bse}).round(4)

,coef,se
Intercept,4.6857,1.3661
C(region)[T.Metro],-5.5405,0.6027
C(region)[T.Other],-6.6684,0.5971
age,-0.0734,0.0300
income,0.1777,0.1283
tenure_months,0.0256,0.0127
n_products,0.5916,0.1591
savings,-0.0047,0.0215
investments,0.0051,0.0052


<div dir="rtl" style="text-align: right;">

### درجه‌ی آزادی باقی‌مانده قبل و بعد از اصلاح

- **قبل از اصلاح:** اگر ستون‌ها را بشماریم، `n − p = 400 − 10 = 390`. ولی رتبه‌ی واقعی ۹ است، پس درجه‌ی آزادی واقعی `n − rank = 391` است. نرم‌افزار هم ۳۹۱ گزارش می‌کند، چون بر اساس رتبه حساب می‌کند.
- **بعد از اصلاح:** `p = rank = 9` و `df = 400 − 9 = 391`.

درجه‌ی آزادی در هر دو حالت ۳۹۱ است. فرق در تفسیر است: قبل از اصلاح، ۱۰ پارامتر داریم ولی فقط ۹ پارامتر مستقل قابل برآورد است. بعد از اصلاح، هر ۹ پارامتر یکتا برآورد می‌شوند.

### چه چیزی عوض شد و چه چیزی نه

- RSS و R² یکسان‌اند. پس قدرت پیش‌بینی مدل با حذف `total_assets` کمتر نشد. این همان نتیجه‌ی بخش قبل است: فقط ضرایب نامعین بودند، نه برازش.
- ضریب‌های مدل خام (مثل `savings` و `total_assets`) بی‌معنی بودند. در مدل اصلاح‌شده یکتا هستند.
- عدد شرط به‌شدت کاهش پیدا کرد. هنوز `income` و `savings` همخطی شدید دارند و این در بخش بعد (VIF) بررسی می‌شود.

</div>